<a href="https://colab.research.google.com/github/deepikasai-mettu/Discriminative-Deep-Learning/blob/main/Demo_MileStone3_YOLOv8_Model_Train_and_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**# IE7615 Project Group 3 - YOLOv8 Models for Discrimitive Project (Fall 2025)**


* Milestone 2 Model initially
* Interim Testing captured
*Generation of collages for multi-face detection
*Milestone 3 Demo model and simple UI

***Deepikasai Mettu, Rediet Merra Tegegne, Suja Ganesh Murugan, Vikas Kagawad***


**Initial Model - Milestone 2 **

In [ ]:
# @title
#YOLOV8 CELEBRITY DETECTION WITH GPU - GOOGLE COLAB

# Step 1: Check GPU (runtime needs to be changed to GPU on colab)
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Step 2: Mount Google Drive (To save variables and access image files)
from google.colab import drive
drive.mount('/content/drive')

# Step 3: Install ultralytics
!pip install -q ultralytics

# Step 4: Unzip dataset (dataset of images + augmented images)
import os
import zipfile

ZIP_PATH = '/content/drive/MyDrive/yolo_celeba_dataset_enhanced.zip'
EXTRACT_TO = '/content/dataset'

print("Extracting dataset...")
os.makedirs(EXTRACT_TO, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_TO)

print("Dataset extracted")

# Step 5: Find and update data.yaml
print("\nFinding data.yaml...")
yaml_path = None

for root, dirs, files in os.walk(EXTRACT_TO):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        print(f"Found: {yaml_path}")
        break

if not yaml_path:
    raise FileNotFoundError("data.yaml not found!")

# Update yaml with correct paths
import yaml

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Update to absolute path
data['path'] = os.path.dirname(yaml_path)

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print(f"Updated data.yaml path: {data['path']}")

# Verify dataset structure
print(f"\nDataset structure:")
print(f"  Train images: {len(os.listdir(os.path.join(data['path'], 'images/train')))}")
print(f"  Val images: {len(os.listdir(os.path.join(data['path'], 'images/val')))}")

# Step 6: Train
from ultralytics import YOLO

# OPTIMIZE to avoid RAM error out

from ultralytics import YOLO

model = YOLO('yolov8n.pt')

print("\n" + "="*70)
print("TRAINING WITH GPU - Memory Optimized")
print("="*70)

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=416,
    batch=16,
    device=0,
    project='/content/runs',
    name='celeb_detector',
    patience=15,
    save=True,
    plots=True,
    verbose=True,
    amp=True,
    cache=False,
    workers=2
)

print("\nTraining complete!")
print("="*70)

# Load class mapping
import pandas as pd

mapping_path = os.path.join(data['path'], 'class_mapping.csv')
class_mapping = pd.read_csv(mapping_path)
reverse_map = dict(zip(class_mapping['class_id'], class_mapping['celebrity_id']))

print(f"Loaded {len(class_mapping)} celebrity mappings")

# Test and visualize
trained_model = YOLO('/content/runs/celeb_detector/weights/best.pt')

val_img_dir = os.path.join(data['path'], 'images/val')
test_imgs = [os.path.join(val_img_dir, f) for f in os.listdir(val_img_dir)[:3]]

print("\nTesting on sample images:")
for img_path in test_imgs:
    results = trained_model(img_path)

    for result in results:
        boxes = result.boxes
        print(f"\n{os.path.basename(img_path)}:")
        print(f"  Detected {len(boxes)} celebrities:")

        for box in boxes:
            class_id = int(box.cls[0].item())
            conf = box.conf[0].item()
            celeb_id = reverse_map[class_id]
            print(f"    - Celebrity ID {celeb_id}: {conf:.2%}")

        # Save visualization
        save_path = f'/content/result_{os.path.basename(img_path)}'
        result.save(save_path)

# Display results
from IPython.display import Image, display
import matplotlib.pyplot as plt

print("\nVisualizing detections:")
for img_path in test_imgs:
    result_path = f'/content/result_{os.path.basename(img_path)}'
    if os.path.exists(result_path):
        display(Image(result_path))

# Save model to Google Drive
import shutil

best_model = '/content/runs/celeb_detector/weights/best.pt'
drive_path = '/content/drive/MyDrive/celeb_detector_best.pt'

shutil.copy(best_model, drive_path)
print(f"\nModel saved to Google Drive: celeb_detector_best.pt")

# Print final metrics
print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
metrics = trained_model.val()
print(f"mAP50: {metrics.box.map50:.3f} ({metrics.box.map50*100:.1f}%)")
print(f"mAP50-95: {metrics.box.map:.3f} ({metrics.box.map*100:.1f}%)")

In [ ]:
# @title
# TEST MODEL
from ultralytics import YOLO
import pandas as pd
import os
from IPython.display import Image, display

# Load trained model
trained_model = YOLO('/content/runs/celeb_detector2/weights/best.pt')

# Load class mapping (celeb ID given 47 sequential numbers)
mapping_path = os.path.join(data['path'], 'class_mapping.csv')
class_mapping = pd.read_csv(mapping_path)
reverse_map = dict(zip(class_mapping['class_id'], class_mapping['celebrity_id']))

print("="*70)
print("TESTING TRAINED MODEL")
print("="*70)

# Validate on entire validation set

print("\n[1] Running validation...")
metrics = trained_model.val()
print(f"\nValidation Results:")
print(f"  mAP50: {metrics.box.map50:.3f} ({metrics.box.map50*100:.1f}%)")
print(f"  mAP50-95: {metrics.box.map:.3f} ({metrics.box.map*100:.1f}%)")

# TEST 2: Test on sample images
# ============================================================================
print("\n[2] Testing on sample images...")

val_img_dir = os.path.join(data['path'], 'images/val')
test_imgs = sorted([os.path.join(val_img_dir, f) for f in os.listdir(val_img_dir) if f.endswith('.jpg')])[1:10]

for img_path in test_imgs:
    print(f"\n--- {os.path.basename(img_path)} ---")

    results = trained_model(img_path, conf=0.25)

    for result in results:
        boxes = result.boxes
        print(f"Detected {len(boxes)} celebrities:")

        for box in boxes:
            class_id = int(box.cls[0].item())
            conf = box.conf[0].item()
            celeb_id = reverse_map[class_id]
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            print(f"  - Celebrity ID {celeb_id}: {conf:.2%} at [{int(x1)}, {int(y1)}, {int(x2)}, {int(y2)}]")

        save_path = f'/content/result_{os.path.basename(img_path)}'
        result.save(save_path)

# TEST 3: Visualize with boxing around objects detected
# ============================================================================
print("\n[3] Visualizing detections...")

for img_path in test_imgs:
    result_path = f'/content/result_{os.path.basename(img_path)}'
    if os.path.exists(result_path):
        print(f"\n{os.path.basename(img_path)}:")
        display(Image(result_path))

print("\n" + "="*70)
print("TESTING COMPLETE!")
print("="*70)

Single Image test

In [ ]:
# @title
# Mount Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

# Load model and test
from ultralytics import YOLO

# model path from Google Drive
model = YOLO('/content/runs/celeb_detector/weights/best.pt')
#/content/runs/detect/val2/

# Test on single image
results = model.predict(
    source='/content/drive/MyDrive/yolo_celeba_dataset/images/train/train_00021.jpg',
    conf=0.34,
    save=True
)


# Results save to: runs/detect/predict/

In [ ]:
# Save model to Google Drive to ensure it can be called in new runtime
import shutil

best_model = '/content/runs/celeb_detector/weights/best.pt'
drive_path = '/content/drive/MyDrive/celeb_detector_best.pt'

shutil.copy(best_model, drive_path)
print(f"\nModel saved to Google Drive: celeb_detector_best.pt")

In [ ]:
# @title
"""
Make Collages of images ~10celebs per TA's request for demo
Tested on Model
"""

import os, random
from pathlib import Path
from typing import List, Optional, Tuple
from PIL import Image, ImageOps
import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
BASE_DIR = Path("/content/drive/MyDrive/celeba_augmented")   # Images
OUTPUT_DIR = Path("./multi_celebrity_examples")  #output folder to save to

N_COLLAGES = 6
FACES_PER_COLLAGE = 12
SEED = 41

# canvas/layout
CANVAS_SIZE: Tuple[int, int] = (640, 640)   # (W,H)
TILE_RANGE = (180, 240)                     # start big
SHRINK_FACTOR = 0.6                          # shrink if overlapping
MIN_TILE = 110                               # min size
MAX_TRIES_PER_TILE = 80                      # max attempts
MIN_GAP = 8                                  # px gap between tiles (ensure no overlap)
EDGE_MARGIN = 4                              # ensures images arent cut off
BG_COLOR = (96, 96, 96)                      # gray background for better contrast

# class map
CLASS_MAP_CSV = Path("/content/drive/MyDrive/class_mapping.csv")

# Trained model (.pt)
MODEL_PATH = Path("/content/drive/MyDrive/celeb_detector_best.pt")

# Parameters for model
CONF = 0.25
IOU = 0.45
MAX_DET = 300
DEVICE = 0   # GPU


# -----------------------------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def gather_images(folder: Path) -> List[Path]:
    return [p for p in folder.rglob("*") if p.suffix.lower() in IMG_EXTS and not p.name.startswith("._")]

def safe_load_image(path: Path) -> Optional[Image.Image]:
    try:
        img = Image.open(path)
        img.load()
        return img.convert("RGB")
    except Exception as e:
        print(f"Skipping unreadable: {path} ({e})")
        return None

def make_square_crop(img: Image.Image, size: int) -> Image.Image:
    """Square crop + resize + no padding"""
    return ImageOps.fit(img, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))

def _rects_overlap(a, b, gap: int) -> bool:
    """>= gap apart to be considered non-overlapping)"""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    return not (ax + aw + gap <= bx or bx + bw + gap <= ax or ay + ah + gap <= by or by + bh + gap <= ay)

def build_scatter_collage_strict(
    image_paths: List[Path],
    out_path: Path,
    faces_per_collage: int,
    canvas_size: Tuple[int, int] = CANVAS_SIZE,
    tile_range: Tuple[int, int] = TILE_RANGE,
    min_tile: int = MIN_TILE,
    shrink_factor: float = SHRINK_FACTOR,
    max_tries: int = MAX_TRIES_PER_TILE,
    min_gap: int = MIN_GAP,
    edge_margin: int = EDGE_MARGIN,
    bg_color: Tuple[int, int, int] = BG_COLOR,
):
    W, H = canvas_size
    canvas = Image.new("RGB", (W, H), bg_color)
    placed_boxes = []  # (x,y,w,h)
    placed = 0

    for p in image_paths:
        if placed >= faces_per_collage:
            break

        img = safe_load_image(p)
        if img is None:
            continue

        # start at a random size in range, then shrink if needed
        tile = random.randint(tile_range[0], tile_range[1])

        for _ in range(max_tries):
            # ensure we don't go below min_tile
            tile = max(tile, min_tile)

            thumb = make_square_crop(img, tile)

            # sample a position
            x = random.randint(edge_margin, max(edge_margin, W - tile - edge_margin))
            y = random.randint(edge_margin, max(edge_margin, H - tile - edge_margin))

            candidate = (x, y, tile, tile)
            if not any(_rects_overlap(candidate, b, min_gap) for b in placed_boxes):
                canvas.paste(thumb, (x, y))
                placed_boxes.append(candidate)
                placed += 1
                break
            else:
                # shrink slightly and try again
                new_tile = int(tile * shrink_factor)
                if new_tile < min_tile:
                    # discard if can't fit
                    break
                tile = new_tile

    out_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out_path, quality=95)
    print(f"Saved collage: {out_path} (placed {placed}/{faces_per_collage})")

def make_collages_from_folder(base_dir: Path):
    if SEED is not None:
        random.seed(SEED)

    images = gather_images(base_dir)
    if not images:
        print(f"No images in folder: {base_dir}")
        return

    random.shuffle(images)
    for i in range(1, N_COLLAGES + 1):
        # oversample a bit to have room to skip/resize problematic images
        picks = random.sample(images, min(FACES_PER_COLLAGE * 3, len(images)))
        out_path = OUTPUT_DIR / "collages" / f"collage_{i:02d}.jpg"
        build_scatter_collage_strict(
            picks,
            out_path,
            faces_per_collage=FACES_PER_COLLAGE,
        )

# MODEL LOADING + TESTING
# -----------------------------
def load_class_map(csv_path: Optional[Path]) -> Optional[pd.DataFrame]:
    if not csv_path or not csv_path.exists():
        print(f"class map CSV not found at {csv_path}")
        return None
    try:
        df = pd.read_csv(csv_path)
        print(f"Loaded class map: {csv_path} (rows={len(df)})")
        return df
    except Exception as e:
        print(f"Failed to read class map: {e}")
        return None

def run_inference_on_folder(model_path: Path, folder: Path, out_dir: Path,
                            conf=CONF, iou=IOU, max_det=MAX_DET, device=DEVICE):
    """Run all images in 'folder' and save predictions."""
    from ultralytics import YOLO

    if not model_path.exists():
        raise FileNotFoundError(f"Model not found: {model_path}")

    model = YOLO(str(model_path))
    out_dir.mkdir(parents=True, exist_ok=True)

    imgs = [p for p in folder.rglob("*") if p.suffix.lower() in IMG_EXTS]
    if not imgs:
        print(f"No images to infer in {folder}")
        return

    print(f"Running detection prediction on {len(imgs)} image(s) from {folder}")
    model.predict(
        source=[str(p) for p in imgs],
        save=True,
        conf=conf,
        iou=iou,
        max_det=max_det,
        device=device,
        project=str(out_dir),
        name="preds",
        exist_ok=True,
        verbose=False
    )
    print(f"Predictions saved under: {out_dir/'preds'}")

# MAIN
# -----------------------------
def main():
    # 1) Build non-overlapping collages of ~10images
    make_collages_from_folder(BASE_DIR)

    # 2) Load class map
    _ = load_class_map(CLASS_MAP_CSV)

    # 3) Inference on collages
    collages_dir = OUTPUT_DIR / "collages"
    preds_dir = OUTPUT_DIR / "detections_collages"
    run_inference_on_folder(MODEL_PATH, folder=collages_dir, out_dir=preds_dir)

if __name__ == "__main__":
    main()


Testing again

In [ ]:
# @title
# Test YOLO on 10 images from yolo_celeba_dataset/images/train

import os, random
from pathlib import Path
import pandas as pd
from ultralytics import YOLO
from IPython.display import Image, display

# ---- paths (edit if yours differ)
DATASET_DIR = Path("/content/drive/MyDrive/yolo_celeba_dataset")
TRAIN_DIR   = DATASET_DIR / "images" / "train"
MODEL_PATH  = Path("/content/drive/MyDrive/celeb_detector_best.pt")   # or your Drive path
OUT_DIR     = Path("./train_eval_out")

# ---- pick 10 images
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
all_imgs = [p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in exts]
assert all_imgs, f"No images found in {TRAIN_DIR}"
random.seed(42)
sampled = random.sample(all_imgs, min(10, len(all_imgs)))

# ---- optional: load class mapping to map class_id -> celeb_id
rev_map = None
map_csv = DATASET_DIR / "class_mapping.csv"
if map_csv.exists():
    dfmap = pd.read_csv(map_csv)
    rev_map = dict(zip(dfmap["class_id"], dfmap["celebrity_id"]))

# ---- load model and run predictions
model = YOLO(str(MODEL_PATH))
OUT_DIR.mkdir(parents=True, exist_ok=True)
preds_dir = OUT_DIR / "train10_preds"

res = model.predict(
    source=[str(p) for p in sampled],
    save=True,
    conf=0.25,
    iou=0.45,
    max_det=300,
    project=str(preds_dir),
    name="yolo",
    exist_ok=True,
    device=0  # set 'cpu' if no GPU
)

# ---- print a compact summary
for r in res:
    img_name = Path(r.path).name
    print(f"\n{img_name}: {len(r.boxes)} detections")
    for b in r.boxes:
        cls_id = int(b.cls[0].item())
        conf   = float(b.conf[0].item())
        celeb  = rev_map.get(cls_id, cls_id) if rev_map is not None else cls_id
        xyxy   = [int(v) for v in b.xyxy[0].tolist()]
        print(f"  - class {cls_id} (celeb {celeb})  conf {conf:.2f}  box {xyxy}")

# ---- show the saved visualizations
vis_folder = preds_dir / "yolo"
if vis_folder.exists():
    to_show = [p for p in vis_folder.iterdir() if p.suffix.lower() in exts]
    for p in to_show:
        display(Image(filename=str(p)))
else:
    print(f"No visualization folder found at {vis_folder}")


**Milestone 3: Improved Model to handle small images (initial model showed it had issues with smaller faces)**

In [ ]:
# @title
# YOLOV8 CELEBRITY DETECTION WITH GPU — edited for Milestone 3 for SMALL FACES

# Step 1: Check GPU
import torch, os, zipfile, yaml, pandas as pd
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 3: Install ultralytics
!pip install -q ultralytics

# ----------------  UNZIP data ----------------
ZIP_PATH = '/content/drive/MyDrive/yolo_celeba_dataset_enhanced.zip'
EXTRACT_TO = '/content/dataset'

print("Extracting dataset...")
os.makedirs(EXTRACT_TO, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_TO)
print("Dataset extracted")

print("\nFinding data.yaml...")
yaml_path = None
for root, dirs, files in os.walk(EXTRACT_TO):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        print(f"Found: {yaml_path}")
        break
if not yaml_path:
    raise FileNotFoundError("data.yaml not found!")

# Update yaml 'path' to absolute
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)
data['path'] = os.path.dirname(yaml_path)
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)
print(f"Updated data.yaml path: {data['path']}")

# Verify dataset structure
train_ct = len(os.listdir(os.path.join(data['path'], 'images/train')))
val_ct   = len(os.listdir(os.path.join(data['path'], 'images/val')))
print(f"\nDataset structure:\n  Train images: {train_ct}\n  Val images:   {val_ct}")

# ---------------- TRAIN ----------------
from ultralytics import YOLO

# Trying YOLOv8m instead of n - research showed it may be better suited for this
model_name = 'yolov8m.pt'
try:
    model = YOLO(model_name)
except Exception as e:
    print(f"Could not load {model_name}. Falling back to yolov8n.pt. ({e})")
    model = YOLO('yolov8n.pt')

print("\n" + "="*70)
print("TRAINING WITH GPU — tuned for small/low-res faces")
print("="*70)

results = model.train(
    data=yaml_path,
    epochs=20,              # lowered to maintain time efficiency
    imgsz=768,             # imcreased image size helps tiny faces
    batch=-1,
    device=0,
    project='/content/runs',
    name='celeb_detector_v2',
    patience=6,
    save=True,
    plots=True,
    verbose=True,
    amp=True,
    cache=False,
    workers=4,
    # --- small object friendly augs ---
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
    scale=0.5,
    degrees=0.0,
    shear=0.0,
    translate=0.05,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    cos_lr=True,
    close_mosaic=10
)

print("\n Training complete!")
print("="*70)

# ---------------- LOAD CLASS MAP ----------------
mapping_path = os.path.join(data['path'], 'class_mapping.csv')
class_mapping = pd.read_csv(mapping_path)
reverse_map = dict(zip(class_mapping['class_id'], class_mapping['celebrity_id']))
print(f"Loaded {len(class_mapping)} celebrity mappings")

# ---------------- TESTING ----------------
trained_path = '/content/runs/celeb_detector/weights/best.pt'
trained_model = YOLO(trained_path)

# Validate once to get mAP50 and mAP95 metrics
print("\nRunning val metrics at imgsz=1024…")
metrics = trained_model.val(imgsz=1024)
print(f"mAP50: {metrics.box.map50:.3f} ({metrics.box.map50*100:.1f}%)")
print(f"mAP50-95: {metrics.box.map:.3f} ({metrics.box.map*100:.1f}%)")

# Sample predictions on a few VAL images
val_img_dir = os.path.join(data['path'], 'images/val')
sample_imgs = [os.path.join(val_img_dir, f) for f in os.listdir(val_img_dir)[:3]]

print("\nTesting on sample images (imgsz=1024, conf=0.15, agnostic_nms=True):")
for img_path in sample_imgs:
    results = trained_model.predict(
        source=img_path, imgsz=768,
        conf=0.15, iou=0.50,
        agnostic_nms=True, max_det=500, device=0, save=True, project='/content', name='preds', exist_ok=True
    )
    # print classes
    for r in results:
        print(f"\n{os.path.basename(img_path)}: {len(r.boxes)} detections")
        for b in r.boxes:
            cid = int(b.cls[0].item())
            conf = b.conf[0].item()
            celeb_id = reverse_map.get(cid, cid)
            print(f"  - Celebrity ID {celeb_id}: {conf:.2%}")

# Display results
from IPython.display import Image, display
for img_path in sample_imgs:
    out = f"/content/preds/{os.path.basename(img_path)}"
    if os.path.exists(out):
        display(Image(out))

# Save model to Google Drive
import shutil
drive_path = '/content/drive/MyDrive/celeb_detector_best.pt'
shutil.copy(trained_path, drive_path)
print(f"\n✓ Model saved to Google Drive: celeb_detector_best.pt")


**Simple Image loading UI for Demo**

In [ ]:
# @title
#!pip install ultralytics -q
# Simple UI to run YOLO on one image (path or filename) with measures to reduce double-detection

import os
from pathlib import Path
from typing import Optional, List, Dict
import pandas as pd
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as w
from ultralytics import YOLO

# paths
MODEL_PATH = Path("/content/drive/MyDrive/celeb_detector_best.pt")
CLASS_MAP_CSV = Path("/content/drive/MyDrive/class_mapping.csv")
SEARCH_DIRS = [
    Path("./yolo_celeba_dataset/images/val"),
    Path("./yolo_celeba_dataset/images/train"),
    Path("./multi_celebrity_examples/collages"),
    Path("."),
]
# Mount Google Drive to get saved model
from google.colab import drive
drive.mount('/content/drive')

# load model
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
model = YOLO(str(MODEL_PATH))

# mapping: class_id -> celeb_id
rev_map = None
if CLASS_MAP_CSV.exists():
    dfm = pd.read_csv(CLASS_MAP_CSV)
    rev_map = dict(zip(dfm["class_id"], dfm["celebrity_id"]))
    print(f"Loaded mapping: {len(rev_map)} classes")
else:
    print(f"CSV not found at {CLASS_MAP_CSV}")
# -------------- Admin random stuff like case sensitivity etc... --------------
def resolve_path(s: str) -> Optional[Path]:
    p = Path(s)
    if p.exists():
        return p
    for d in SEARCH_DIRS:
        cand = d / s
        if cand.exists():
            return cand
    # case-insensitive fallback
    name_lower = Path(s).name.lower()
    for d in SEARCH_DIRS:
        if not d.exists():
            continue
        for f in d.iterdir():
            try:
                if f.is_file() and f.name.lower() == name_lower:
                    return f
            except Exception:
                pass
    return None

def predict_one_image(p: Path, conf=0.15, iou=0.40, imgsz=640, device=0, tta=True):
    # agnostic_nms=True = to suppress overlaps across classes
    return model.predict(
        source=str(p),
        conf=conf,
        iou=iou,
        imgsz=imgsz,
        augment=tta,
        agnostic_nms=True,
        device=device,
        save=True,
        verbose=False
    )

#  cross-class deduplication (extra measure against double-detection on one face)
def _iou_xyxy(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    return inter / (area_a + area_b - inter + 1e-9)

def dedup_across_classes(dets: List[Dict], iou_thresh=0.5) -> List[Dict]:
    # dets: [{"cls": int, "conf": float, "xyxy": [x1,y1,x2,y2]}]
    dets = sorted(dets, key=lambda d: d["conf"], reverse=True)
    kept = []
    for d in dets:
        if all(_iou_xyxy(d["xyxy"], k["xyxy"]) < iou_thresh for k in kept):
            kept.append(d)
    return kept

# -------------- UI --------------

# -------------- Styled UI --------------
from IPython.display import HTML
display(HTML("""
<style>
    .maroon-theme .widget-text input {
        background-color: #fff5f5 !important;
        border: 2px solid #800020 !important;
        border-radius: 8px !important;
        padding: 10px !important;
        font-size: 14px !important;
    }

    .maroon-theme .widget-button {
        background-color: #800020 !important;
        color: white !important;
        border: none !important;
        border-radius: 8px !important;
        padding: 12px 24px !important;
        font-weight: bold !important;
        font-size: 14px !important;
        box-shadow: 0 4px 6px rgba(128, 0, 32, 0.3) !important;
    }

    .maroon-theme .widget-button:hover {
        background-color: #a0002a !important;
        box-shadow: 0 6px 12px rgba(128, 0, 32, 0.4) !important;
    }

    .maroon-theme .widget-label {
        color: #333 !important;
        font-weight: 600 !important;
    }

    .maroon-theme .widget-slider .slider {
        background: #800020 !important;
    }

    .maroon-theme .widget-dropdown select {
        background-color: #fff5f5 !important;
        border: 2px solid #800020 !important;
        border-radius: 6px !important;
        padding: 5px !important;
    }

    .project-header {
        background: linear-gradient(135deg, #800020 0%, #a0002a 100%);
        color: white;
        padding: 20px;
        border-radius: 10px;
        text-align: center;
        font-size: 24px;
        font-weight: bold;
        margin-bottom: 20px;
        box-shadow: 0 4px 8px rgba(0,0,0,0.2);
    }
</style>
"""))

# Header
header = w.HTML(value='<div class="project-header">IE7615 Project 1, Group 3</div>')

# Widgets for UI for demo 10/16
inp   = w.Text(placeholder="full path or filename (e.g., val_00010.jpg)", description="Image:", layout=w.Layout(width="70%"))
confW = w.FloatSlider(value=0.15, min=0.05, max=0.50, step=0.01, description="Confidence", readout_format=".2f", style={'description_width': '80px'})
iouW  = w.FloatSlider(value=0.40, min=0.30, max=0.70, step=0.01, description="IoU", readout_format=".2f", style={'description_width': '80px'})
szW   = w.Dropdown(options=[416, 512, 640, 736, 832], value=640, description="Image Size", style={'description_width': '80px'})
ttaW  = w.Checkbox(value=True, description="TTA", style={'description_width': 'initial'})
devW  = w.Dropdown(options=[0, 'cpu'], value=0, description="Device", style={'description_width': '80px'})
btn   = w.Button(description="Run", button_style="",layout=w.Layout(min_width='80px'))
out   = w.Output()

def on_click(_):
    with out:
        clear_output()
        name = inp.value.strip()
        if not name:
            print("Enter a path or filename.")
            return

        p = resolve_path(name)
        if not p:
            print("Not found. Checked:")
            for d in SEARCH_DIRS: print(" -", d)
            return

        print(f"Image: {p}")

        print("ORIGINAL IMAGE:")
        print("-"*50)
        try:
            display(Image.open(p).convert("RGB"))
        except Exception:
            pass

        res = predict_one_image(
            p, conf=confW.value, iou=iouW.value, imgsz=szW.value, device=devW.value, tta=ttaW.value
        )
        if not res:
            print("No results.")
            return

        r = res[0]

        # collect raw detections
        raw = []
        for b in r.boxes:
            cls_id = int(b.cls[0].item())
            confv  = float(b.conf[0].item())
            xyxy   = [int(v) for v in b.xyxy[0].tolist()]
            raw.append({"cls": cls_id, "conf": confv, "xyxy": xyxy})

        # keeps top-confidence per face and not duplicates
        dedup = dedup_across_classes(raw, iou_thresh=0.50)


        print(f"RESULTS:")
        print("-"*50)
        print(f"Detections (raw): {len(raw)}")
        print(f"Detections (after de-duplication): {len(dedup)}")
        print()

        for i, d in enumerate(dedup, 1):
            if rev_map is not None:
                celeb_id = rev_map.get(d["cls"], f"Unknown-{d['cls']}")
                print(f"{i}. Celebrity ID: {celeb_id} | Class: {d['cls']} | Confidence: {d['conf']:.2%} | Box: {d['xyxy']}")
            else:
                print(f"{i}. Class ID: {d['cls']} | Confidence: {d['conf']:.2%} | Box: {d['xyxy']}")

        # show YOLO visualization with boxes
        print("IMAGE WITH BOUNDING BOXES:")
        print("-"*50)

        # save path from results object
        if hasattr(r, 'save_dir') and r.save_dir:
            save_path = Path(r.save_dir) / p.name
            if save_path.exists():
                display(Image.open(save_path).convert("RGB"))
            else:
                print(f"Visualization not found at: {save_path}")
        else:
            # alternatively search in runs directory
            try:
                runs_dir = Path("runs/detect")
                if runs_dir.exists():
                    pred_folders = sorted([f for f in runs_dir.iterdir() if f.is_dir() and f.name.startswith('predict')],
                                        key=os.path.getmtime, reverse=True)
                    if pred_folders:
                        vis = pred_folders[0] / p.name
                        if vis.exists():
                            display(Image.open(vis).convert("RGB"))
                        else:
                            print(f"Visualization not found at: {vis}")
                    else:
                        print("No prediction folders found")
                else:
                    print("runs/detect directory not found")
            except Exception as e:
                print(f"Error loading viz: {e}")

btn.on_click(on_click)
controls = w.HBox([confW, iouW, szW, ttaW, devW, btn])
ui = w.VBox([header, inp, controls, out])
ui.add_class('maroon-theme')
display(ui)

**Concatenate images with >=10 celebs per TA's request for 10/16 Demo**

In [ ]:
# @title
# Concatenate random non-overlapping collages (10–12 faces) for the 47 target CelebA IDs

# !pip install -q kagglehub  # uncomment if needed

import os, random
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import pandas as pd
from PIL import Image, ImageOps
import kagglehub

# ---------------- config ----------------
# how many composites to make
N_COMPOSITES = 6
FACES_PER_COMPOSITE = (10, 12)  # min, max (inclusive)

# canvas and tile placement
CANVAS_SIZE: Tuple[int, int] = (640, 640)  # (W, H)
TILE_RANGE = (110, 180)      # starting tile size; will shrink if needed
MIN_TILE   = 70              # hard floor for tile size
SHRINK     = 0.88            # shrink factor when a placement fails
MAX_TRIES  = 150              # attempts per face
MARGIN     = 6               # gap between tiles
EDGE_PAD   = 4               # keep away from edges
BG_COLOR   = (96, 96, 96)    # grey background
SEED       = 42

# save here
OUT_DIR = Path("./celeba_47_collages")
SAVE_YOLO_LABELS = True  # also write YOLO txt labels next to each image

# your 47 identities (as given)
TARGET_IDS = [
    4126, 7904, 8656, 9319, 3321, 8968, 2820, 3227, 9063, 8871,
    7282, 8945, 3782, 8722, 3401, 1964, 4561, 2880, 3745, 3699,
    9152, 9256, 2463, 2562, 3431, 1499, 8045, 10173, 2522, 228,
    5239, 2425, 4304, 5260, 2837, 1158, 3698, 6098, 6568, 9151,
    800, 619, 487, 1852, 8265, 447, 1757
]
# ----------------------------------------

random.seed(SEED)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Get CelebA and paths
print("Downloading CelebA (first run only)…")
root = Path(kagglehub.dataset_download("jessicali9530/celeba-dataset"))

img_dirs = [
    root / "img_align_celeba" / "img_align_celeba",
    root / "img_align_celeba"
]
IMG_DIR = next((p for p in img_dirs if p.is_dir()), None)
assert IMG_DIR is not None, "Could not locate 'img_align_celeba' folder."

# identity file
id_file = Path("/content/drive/MyDrive/identity_CelebA.txt")
assert id_file.exists(), "identity_CelebA.txt not found in dataset."

df = pd.read_csv(id_file, sep=r"\s+", header=None, names=["filename", "identity"])
df47 = df[df["identity"].isin(TARGET_IDS)].copy()
assert len(df47), "No images found for the given IDs."

# class mapping (0..N-1)
id_to_class = {id_: i for i, id_ in enumerate(sorted(df47["identity"].unique()))}
class_to_id = {v: k for k, v in id_to_class.items()}
num_classes = len(id_to_class)
print(f"Using {num_classes} identities with {len(df47):,} images.")

# build index: identity -> list of filenames
id_to_files: Dict[int, List[str]] = (
    df47.groupby("identity")["filename"].apply(list).to_dict()
)

# helpers
def safe_open(path: Path) -> Optional[Image.Image]:
    try:
        im = Image.open(path)
        im.load()
        return im.convert("RGB")
    except Exception:
        return None

def square_resize(img: Image.Image, size: int) -> Image.Image:
    # center-crop to square then resize; avoids bars
    return ImageOps.fit(img, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))

def overlap(a, b, gap: int) -> bool:
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    return not (ax + aw + gap <= bx or bx + bw + gap <= ax or ay + ah + gap <= by or by + bh + gap <= ay)

def place_faces_random(
    identities: List[int],
    canvas_size: Tuple[int, int] = CANVAS_SIZE
) -> Tuple[Image.Image, List[Tuple[int, float, float, float, float]]]:
    """Return composite PIL image and YOLO boxes: (class_id, xc, yc, w, h)."""
    W, H = canvas_size
    canvas = Image.new("RGB", (W, H), BG_COLOR)
    placed_boxes = []  # (x,y,w,h) pixel
    yolo_boxes = []    # (cls, xc, yc, w, h) normalized

    for ident in identities:
        files = id_to_files.get(ident, [])
        if not files:
            continue
        fp = IMG_DIR / random.choice(files)
        img = safe_open(fp)
        if img is None:
            continue

        tile = random.randint(*TILE_RANGE)
        ok = False
        for _ in range(MAX_TRIES):
            tile = max(tile, MIN_TILE)
            thumb = square_resize(img, tile)

            x = random.randint(EDGE_PAD, max(EDGE_PAD, W - tile - EDGE_PAD))
            y = random.randint(EDGE_PAD, max(EDGE_PAD, H - tile - EDGE_PAD))
            cand = (x, y, tile, tile)

            if not any(overlap(cand, b, MARGIN) for b in placed_boxes):
                # paste and record
                canvas.paste(thumb, (x, y))
                placed_boxes.append(cand)

                # YOLO normalized (class, x_c, y_c, w, h)
                x_c = (x + tile / 2) / W
                y_c = (y + tile / 2) / H
                w_n = tile / W
                h_n = tile / H
                cls = id_to_class[ident]
                yolo_boxes.append((cls, x_c, y_c, w_n, h_n))
                ok = True
                break
            else:
                new_tile = int(tile * SHRINK)
                if new_tile < MIN_TILE:
                    break
                tile = new_tile

        if not ok:
            # skip this identity if we couldn't place it
            continue

    return canvas, yolo_boxes

# 2) Make collages
saved = []
for i in range(1, N_COMPOSITES + 1):
    k = random.randint(FACES_PER_COMPOSITE[0], FACES_PER_COMPOSITE[1])
    # pick distinct identities each time
    picks = random.sample(TARGET_IDS, min(k, len(TARGET_IDS)))
    img, boxes = place_faces_random(picks, CANVAS_SIZE)

    out_img = OUT_DIR / f"collage_{i:02d}.jpg"
    img.save(out_img, quality=95)
    saved.append(out_img)

    if SAVE_YOLO_LABELS:
        out_txt = OUT_DIR / f"collage_{i:02d}.txt"
        with open(out_txt, "w") as f:
            for cls, xc, yc, w, h in boxes:
                f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

    print(f"saved: {out_img}  (placed {len(boxes)})")

# 3) Optional preview in notebook
try:
    from IPython.display import display
    for p in saved[:5]:
        display(Image.open(p))
except Exception:
    pass

# also save the mapping (class_id -> celeb_id) for reference
pd.DataFrame(
    [{"class_id": c, "celebrity_id": class_to_id[c]} for c in sorted(class_to_id)]
).to_csv(OUT_DIR / "class_mapping_47.csv", index=False)

print("\nDone.")
print(f"Composites folder: {OUT_DIR.resolve()}")
print(f"Classes: {num_classes}")


In [ ]:
# @title
from pathlib import Path
import shutil

RUN_DIR_SRC = Path("/content/runs/celeb_detector")  # folder containing weights, results.png, etc.
RUN_DIR_DST = Path("/content/drive/MyDrive/celebrity_project_outputs/runs/celeb_detector")

if RUN_DIR_SRC.exists():
    for p in RUN_DIR_SRC.rglob("*"):
        if p.is_file():
            rel = p.relative_to(RUN_DIR_SRC)
            (RUN_DIR_DST / rel).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, RUN_DIR_DST / rel)
    print("✓ Copied training artifacts to:", RUN_DIR_DST)
else:
    print("⚠ Run directory not found:", RUN_DIR_SRC)


--10/16 Demo Fancier UI - testing

In [ ]:
#!pip install ultralytics gradio -qq

import gradio as gr
from pathlib import Path
from typing import Optional, List, Dict
import pandas as pd
from PIL import Image
from ultralytics import YOLO
import os

# paths
MODEL_PATH = Path("/content/drive/MyDrive/celeb_detector_best.pt")
CLASS_MAP_CSV = Path("/content/drive/MyDrive/class_mapping.csv")
SEARCH_DIRS = [
    Path("./yolo_celeba_dataset/images/val"),
    Path("./yolo_celeba_dataset/images/train"),
    Path("./multi_celebrity_examples/collages"),
    Path("."),
]

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# load model
model = YOLO(str(MODEL_PATH))

# mapping
rev_map = None
if CLASS_MAP_CSV.exists():
    dfm = pd.read_csv(CLASS_MAP_CSV)
    rev_map = dict(zip(dfm["class_id"], dfm["celebrity_id"]))

def resolve_path(s: str) -> Optional[Path]:
    p = Path(s)
    if p.exists():
        return p
    for d in SEARCH_DIRS:
        cand = d / s
        if cand.exists():
            return cand
    name_lower = Path(s).name.lower()
    for d in SEARCH_DIRS:
        if not d.exists():
            continue
        for f in d.iterdir():
            try:
                if f.is_file() and f.name.lower() == name_lower:
                    return f
            except Exception:
                pass
    return None

def _iou_xyxy(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    return inter / (area_a + area_b - inter + 1e-9)

def dedup_across_classes(dets: List[Dict], iou_thresh=0.5) -> List[Dict]:
    dets = sorted(dets, key=lambda d: d["conf"], reverse=True)
    kept = []
    for d in dets:
        if all(_iou_xyxy(d["xyxy"], k["xyxy"]) < iou_thresh for k in kept):
            kept.append(d)
    return kept

def predict(image_path, conf, iou, imgsz, tta, device):
    p = resolve_path(image_path)
    if not p:
        return None, "Image not found!", None

    # Run prediction
    res = model.predict(
        source=str(p),
        conf=conf,
        iou=iou,
        imgsz=imgsz,
        augment=tta,
        agnostic_nms=True,
        device=device,
        save=True,
        verbose=False
    )

    if not res:
        return str(p), "No detections", None

    r = res[0]

    # Collect detections
    raw = []
    for b in r.boxes:
        cls_id = int(b.cls[0].item())
        confv = float(b.conf[0].item())
        xyxy = [int(v) for v in b.xyxy[0].tolist()]
        raw.append({"cls": cls_id, "conf": confv, "xyxy": xyxy})

    dedup = dedup_across_classes(raw, iou_thresh=0.50)

    # Format results
    results_text = f"Detections (raw): {len(raw)}\n"
    results_text += f"Detections (after dedup): {len(dedup)}\n\n"

    for i, d in enumerate(dedup, 1):
        if rev_map is not None:
            celeb_id = rev_map.get(d["cls"], f"Unknown-{d['cls']}")
            results_text += f"{i}. Celebrity ID: {celeb_id} | Class: {d['cls']} | Confidence: {d['conf']:.2%}\n"
        else:
            results_text += f"{i}. Class: {d['cls']} | Confidence: {d['conf']:.2%}\n"

    # Get visualization
    vis_path = None
    if hasattr(r, 'save_dir') and r.save_dir:
        vis_path = Path(r.save_dir) / p.name
    else:
        runs_dir = Path("runs/detect")
        if runs_dir.exists():
            pred_folders = sorted([f for f in runs_dir.iterdir() if f.is_dir() and f.name.startswith('predict')],
                                key=os.path.getmtime, reverse=True)
            if pred_folders:
                vis_path = pred_folders[0] / p.name

    return str(p), results_text, str(vis_path) if vis_path and vis_path.exists() else None

# Create Gradio interface
with gr.Blocks(theme=gr.themes.Base(), css="""
    .gradio-container {background: linear-gradient(135deg, #800020 0%, #a0002a 100%) !important;}
    .gr-button-primary {background: #800020 !important; border: none !important;}
    h1 {color: white !important; text-align: center !important; padding: 20px !important;}
""", title="IE7615 Project 1, Group 3") as demo:

    gr.Markdown("# IE7615 Project 1, Group 3", elem_classes="header")

    with gr.Row():
        with gr.Column():
            image_input = gr.Textbox(label="Image Path", placeholder="full path (google drive only)")
            with gr.Row():
                conf_slider = gr.Slider(0.05, 0.50, value=0.15, step=0.01, label="Confidence")
                iou_slider = gr.Slider(0.30, 0.70, value=0.40, step=0.01, label="IoU")
            with gr.Row():
                imgsz_dropdown = gr.Dropdown([416, 512, 640, 736, 832], value=640, label="Image Size")
                tta_checkbox = gr.Checkbox(value=True, label="TTA")
                device_dropdown = gr.Dropdown([0, "cpu"], value=0, label="Device")
            run_btn = gr.Button("Run", variant="primary")

    with gr.Row():
        with gr.Column():
            original_path = gr.Textbox(label="Image Path")
            results_output = gr.Textbox(label="Detection Results", lines=10)
        with gr.Column():
            prediction_image = gr.Image(label="Prediction with Bounding Boxes", type="filepath")

    run_btn.click(
        predict,
        inputs=[image_input, conf_slider, iou_slider, imgsz_dropdown, tta_checkbox, device_dropdown],
        outputs=[original_path, results_output, prediction_image]
    )

# Launches in new tab
demo.launch(share=True, inbrowser=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')